# 06 · Delta Live Tables: el pipeline declarativo

⏱ **18 min** · Módulo 5 · Slide de la Intro: 22 (Delta Live Tables: declarativo)

**Objetivo:** construir el mismo flujo bronze → silver → gold **declarando** las tablas en SQL. Databricks resuelve el orden,
la carga incremental y la calidad.

| Imperativo: tú orquestas (notebooks 04 y 05) | Declarativo: tú declaras (este notebook) |
|---|---|
| Escribes cada paso | Declaras tablas y consultas |
| Defines el orden con `depends_on` | Databricks deduce las dependencias |
| Validas con una tarea aparte | Las expectativas de calidad van dentro de la tabla |
| `overwrite`: reprocesas todo | `STREAM`: solo entra lo nuevo |

| Paso | Qué haces |
|---|---|
| 1 | Leer el código del pipeline |
| 2 | Crear el pipeline |
| 3 | Ejecutarlo |
| 4 | Ver las tablas |
| 5 | Expectativas: la calidad de los datos |
| 6 | Llega un archivo con errores |
| 7 | Programarlo |

> **El nombre:** Delta Live Tables aparece hoy en la interfaz como **Lakeflow Spark Declarative Pipelines**
> (menú **Jobs & Pipelines → ETL pipeline**). Es la misma funcionalidad.

**Requiere:** notebook 04.

In [0]:
%run ./00_Setup

## Paso 1 · Leer el código del pipeline

Todo el pipeline es un archivo SQL con tres sentencias.

In [0]:
LAB_DIR = os.getcwd()        # la carpeta de este laboratorio en el Workspace
PIPELINE_FILE = f"{LAB_DIR}/pipeline/tvmaze_pipeline.sql"

print(Path(PIPELINE_FILE).read_text(encoding="utf-8"))

👀 **Qué observar:**

- No hay orden de ejecución, ni `write`, ni `overwrite`: solo **qué** tablas quieres y **cómo** se calculan.
- `STREAMING TABLE` + `STREAM`: la tabla recibe solo los datos nuevos (por dentro, `read_files` es Auto Loader).
- `MATERIALIZED VIEW`: el resultado de una consulta, que Databricks mantiene al día.
- `CONSTRAINT ... EXPECT`: las reglas de calidad, como en el slide 22.

## Paso 2 · Crear el pipeline

In [0]:
pipeline = {
    "name": PIPELINE_NAME,
    "catalog": CATALOG,                              # dónde se publican las tablas: Unity Catalog
    "schema": SCHEMA,
    "serverless": True,
    "development": True,                             # modo desarrollo: si algo falla, no reintenta
    "libraries": [{"file": {"path": PIPELINE_FILE}}],
    "configuration": {"raw_path": RAW_PATH},         # el valor de ${raw_path} en el SQL
}

PIPELINE_ID = create_pipeline(pipeline)

## Paso 3 · Ejecutarlo

Tarda unos minutos. Abre el enlace y mira cómo se arma el gráfico.

In [0]:
run_pipeline(PIPELINE_ID)

👀 **Qué observar:** Databricks leyó las tres sentencias y **dedujo el orden**: `dlt_bronze_schedule → dlt_silver_episodes → dlt_gold_network_daily`.
Nadie escribió un `depends_on`. Cada caja muestra cuántas filas escribió.

## Paso 4 · Ver las tablas

In [0]:
%sql
SELECT table_name, table_type
FROM information_schema.tables
WHERE table_schema = 'lab_tvmaze_m56' AND table_name LIKE 'dlt%'
ORDER BY table_name

In [0]:
%sql
SELECT source_file, count(*) AS episodios
FROM dlt_bronze_schedule
GROUP BY source_file
ORDER BY source_file

In [0]:
%sql
SELECT * FROM dlt_gold_network_daily
ORDER BY episodes DESC
LIMIT 10

👀 **Qué observar:** son tablas normales de Unity Catalog (se gobiernan igual que las del módulo 4), pero las mantiene el pipeline:
dos *streaming tables* y una *materialized view*.

## Paso 5 · Expectativas: la calidad de los datos

Las tres reglas de `dlt_silver_episodes`, con cuántas filas las cumplieron.

In [0]:
show_expectations(PIPELINE_ID)

👀 **Qué observar:** `duracion_positiva` tiene fallos (episodios sin duración), pero esas filas **sí entraron**: esa regla solo avisa.
Los mismos números están en la interfaz del pipeline, al seleccionar la tabla silver.

| Si la regla no se cumple | Sintaxis | Efecto |
|---|---|---|
| Avisar | `EXPECT (...)` | La fila entra y queda la métrica |
| Descartar | `EXPECT (...) ON VIOLATION DROP ROW` | La fila no entra |
| Detener | `EXPECT (...) ON VIOLATION FAIL UPDATE` | La actualización falla |

## Paso 6 · Llega un archivo con errores

Un archivo nuevo en RAW con dos episodios malos: uno sin id y otro con una fecha que no es fecha.

In [0]:
write_bad_file()
run_pipeline(PIPELINE_ID)

In [0]:
show_expectations(PIPELINE_ID)

In [0]:
%sql
SELECT 'bronze' AS capa, count(*) AS filas_del_archivo_malo FROM dlt_bronze_schedule WHERE source_file = 'schedule_con_errores.json'
UNION ALL
SELECT 'silver', count(*) FROM dlt_silver_episodes WHERE source_file = 'schedule_con_errores.json'

👀 **Qué observar:**

- Bronze recibió las 2 filas (guarda todo, tal como llega). Silver descartó ambas: `episodio_con_id` y `fecha_valida` tienen 1 fallo cada una.
- El pipeline no se detuvo ni hubo que escribir una tarea de validación.
- Esta segunda ejecución leyó **solo el archivo nuevo**: en la interfaz, la caja de bronze dice 2 filas. Es incremental sin escribir código.

## Paso 7 · Programarlo

Un pipeline también se ejecuta solo. Dos formas:

- En la interfaz del pipeline: botón **Schedule**.
- Como una tarea más de un Job, junto a otras tareas:

```python
{"task_key": "pipeline_tvmaze", "pipeline_task": {"pipeline_id": PIPELINE_ID}}
```

## ✔ Checkpoint

- [ ] Sé la diferencia entre el enfoque imperativo y el declarativo
- [ ] Creé y ejecuté un pipeline declarativo con tres tablas
- [ ] Sé qué hacen `EXPECT`, `ON VIOLATION DROP ROW` y `ON VIOLATION FAIL UPDATE`
- [ ] Vi que la segunda ejecución procesó solo el archivo nuevo

**Siguiente:** `07_SQL_Warehouse` (empieza el módulo 6)